# 03 — Supervised Fraud Classification

Uses the [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) dataset (ULB Machine Learning Group), which has real fraud labels (`Class`), unlike the bank transactions dataset used in notebooks 01–02.

**Goal:** train a genuine binary classifier and evaluate it with metrics appropriate for a severely imbalanced problem — not raw accuracy, which is misleading here.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

df = pd.read_csv('../data/raw/creditcard.csv')
df.shape

(284807, 31)

## 1. Class imbalance

`V1`–`V28` are PCA-anonymized features (the bank withheld the original variables for privacy); only `Time` and `Amount` are in their original form.


In [2]:
df['Class'].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

In [3]:
fraud_pct = df['Class'].mean()*100
print(f"Fraud rate: {fraud_pct:.3f}%")

# what accuracy would a model get by always predicting "normal"?
dumb_accuracy = (df['Class']==0).mean()*100
print(f"'Always predict normal' accuracy: {dumb_accuracy:.2f}%")

Fraud rate: 0.173%
'Always predict normal' accuracy: 99.83%


A trivial model that never predicts fraud achieves **99.83% accuracy** while catching zero fraud cases. This makes accuracy useless as an evaluation metric here — we use **precision** and **recall** instead:

- **Recall** = of all actual frauds, what fraction did the model catch? (missing this is costly — a genuine loss)
- **Precision** = of everything the model flagged as fraud, what fraction actually was? (missing this means false alarms / customer friction)


## 2. Train/test split

We stratify by `Class` so the fraud rate is preserved in both splits.


In [4]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} rows, {y_train.sum()} fraud ({y_train.mean()*100:.3f}%)")
print(f"Test:  {X_test.shape[0]} rows, {y_test.sum()} fraud ({y_test.mean()*100:.3f}%)")

Train: 227845 rows, 394 fraud (0.173%)
Test:  56962 rows, 98 fraud (0.172%)


## 3. Baseline: Logistic Regression (unweighted)


In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base.fit(X_train_scaled, y_train)
y_pred_base = lr_base.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred_base))
print(classification_report(y_test, y_pred_base, target_names=['Normal','Fraud']))

[[56851    13]
 [   36    62]]
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.83      0.63      0.72        98

    accuracy                           1.00     56962
   macro avg       0.91      0.82      0.86     56962
weighted avg       1.00      1.00      1.00     56962



## 4. Logistic Regression with class balancing

`class_weight='balanced'` penalizes missed-fraud errors more heavily during training, to counteract the imbalance.


In [6]:
lr_bal = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_bal.fit(X_train_scaled, y_train)
y_pred_bal = lr_bal.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred_bal))
print(classification_report(y_test, y_pred_bal, target_names=['Normal','Fraud']))

[[55478  1386]
 [    8    90]]
              precision    recall  f1-score   support

      Normal       1.00      0.98      0.99     56864
       Fraud       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



Recall improves sharply (more fraud caught) but precision collapses — the model now over-flags many legitimate transactions. This is the classic **precision-recall trade-off**: pushed too far toward recall, the false-alarm volume becomes operationally unworkable (a bank cannot manually review thousands of false positives to catch a handful of extra frauds).


## 5. Random Forest

A non-linear model, better suited to capturing complex interactions between the anonymized PCA features than a linear classifier.


In [7]:
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)  # tree-based models don't require scaling
y_pred_rf = rf.predict(X_test)

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, target_names=['Normal','Fraud']))

joblib.dump(rf, '../data/processed/random_forest_model.pkl')

[[56861     3]
 [   25    73]]
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.96      0.74      0.84        98

    accuracy                           1.00     56962
   macro avg       0.98      0.87      0.92     56962
weighted avg       1.00      1.00      1.00     56962



['../data/processed/random_forest_model.pkl']

## 6. Model comparison

| Model | Fraud caught | Fraud missed | False alarms | Precision | Recall |
|---|---|---|---|---|---|
| Logistic Regression (baseline) | 62/98 | 36 | 13 | 0.83 | 0.63 |
| Logistic Regression (balanced) | 90/98 | 8 | 1386 | 0.06 | 0.92 |
| **Random Forest (balanced)** | **73/98** | **25** | **3** | **0.96** | **0.74** |

Random Forest gives the best operational trade-off here: high precision (few false alarms — feasible to actually review manually) with substantially better recall than the unweighted baseline. We use this model going forward.

Next: `04_risk_quantification.ipynb` — translating these classification results into dollar-denominated risk measures (expected loss, Monte Carlo simulation, VaR/CVaR).
